In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras import layers, models
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.applications import MobileNetV2

gpus = tf.config.list_physical_devices('GPU')

if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

(x_train, y_train), (x_test, y_test) = cifar10.load_data()

y_train = tf.keras.utils.to_categorical(y_train, 10)
y_test = tf.keras.utils.to_categorical(y_test, 10)

def preprocess_image(image, label):
    image = tf.image.resize(image, (96, 96))
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

train_dataset = tf.data.Dataset.from_tensor_slices(
    (x_train, y_train)
)

train_dataset = train_dataset.map(
    preprocess_image,
    num_parallel_calls=tf.data.AUTOTUNE
)

train_dataset = train_dataset.shuffle(10000).batch(32).prefetch(
    tf.data.AUTOTUNE
)

test_dataset = tf.data.Dataset.from_tensor_slices(
    (x_test, y_test)
)

test_dataset = test_dataset.map(
    preprocess_image,
    num_parallel_calls=tf.data.AUTOTUNE
)

test_dataset = test_dataset.batch(32).prefetch(
    tf.data.AUTOTUNE
)

base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(96, 96, 3)
)

base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

model.fit(
    train_dataset,
    epochs=2,
    validation_data=test_dataset
)

model.save('mobilenetv2_cifar10.keras')

loss, accuracy = model.evaluate(test_dataset)

print("Test Accuracy:", accuracy)

 24387584/170498071 ━━━━━━━━━━━━━━━━━━━━ 9:32 4us/step

In [ ]:
class_names = [
    'airplane', 'automobile',
    'bird', 'cat', 'deer',
    'dog', 'frog', 'horse',
    'ship','truck'
]

model = tf.keras.models.load_model(
    'mobilenetv2_cifar10.keras'
)

plt.figure(figsize=(10, 10))

for i in range(9):
    image = x_test[i]

    processed_image = tf.image.resize(
        image,
        (96, 96)
    )

    processed_image = tf.cast(
        processed_image,
        tf.float32
    ) / 255.0

    prediction = model.predict(
        tf.expand_dims(processed_image, 0),
        verbose=0
    )

    predicted_class = np.argmax(prediction[0])
    confidence = np.max(prediction[0]) * 100

    plt.subplot(3, 3, i + 1)
    plt.imshow(image)
    plt.title(
        f"{class_names[predicted_class]} "
        f"{confidence:.2f}%"
    )
    plt.axis('off')

plt.tight_layout()
plt.show()